# Welcome to Colab!

In [4]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-google-genai faiss-cpu pypdf

In [12]:

import os

# 1. Configure API key
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# 2. Import libraries
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 3. Find and load PDF
path = (
    "/content/notes.pdf"
    if os.path.exists("/content/notes.pdf")
    else "/notes.pdf"
)

if not os.path.exists(path):
    raise FileNotFoundError("PDF file not found!")

docs = PyPDFLoader(path).load()

if not docs:
    raise ValueError("PDF contains no readable pages!")

# 4. Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(docs)

# 5. Create embeddings and vector database
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

vectorstore = FAISS.from_documents(chunks, embeddings)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

# 6. Create prompt
prompt = ChatPromptTemplate.from_template(
    """Answer only from the context below.
    If the answer is not in the context, say you don't know.

    Context:
    {context}

    Question: {question}"""
)

# 7. Initialize Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    timeout=60,
    max_retries=3
)

response = llm.invoke("Say hello in one word")
print(response.content)

# 8. Test Gemini
print(llm.invoke("Say hello in one word").content)

# 9. Format retrieved documents
def format_docs(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )

# 10. Create RAG chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# 11. Test question
print("Ready! Loaded", len(chunks), "chunks.")

response = rag_chain.invoke("How much is the course fee?")


print(response)


[{'type': 'text', 'text': 'Hello', 'extras': {'signature': 'Eo4BCosBAWkUfRNmVVc0NeHtu7Up3AqvM1T8V3M7uO4pYit4kJ7Y6xTTx0Os57M9rTSOC7ZrnwmrTzClw2b2UEkwluS1DjSxwUcDRZj3h3bcQqNYX12KI8lC1IrLcHcZa98ZpciuEd1zT21v634VJVGvNPs8QbpEEHQxbyMFV4u7gk/RXTp4VMPGb8AFDgZD1A=='}}]
[{'type': 'text', 'text': 'Hello.', 'extras': {'signature': 'Eo4BCosBAWkUfRP7Wcyxc1kS6o7wHymGI0mwakkne1+XGyqbu1VW4n4cQRnfWKfthvAnUSKsqSau2re1wE5aNH9/SvshXcvzSO5jrgW5LAywRlxQ69NB1XnscTnqZtVNNNEcCQjUgIslVFiUzt0Rd+Xp1Xenu/JOEUalvU2ogkpCt+daYrABSdxmniKjJMNGkw=='}}]
Ready! Loaded 4 chunks.
The course fee is Rs. 14,999 including GST.
